In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score



## Get Data


In [ ]:
df = pd.read_csv("superstore_eda_cleaned.csv")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()

## ML Objective

Objective:
Predict the Region of an order using sales value, delivery time, product category, customer segment, and city information.
This helps identify whether regions exhibit distinguishable purchasing and logistics behavior.

In [ ]:
features = [
    'Sales',
    'Delivery_Days',
    'Category',
    'Sub-Category',
    'Segment',
    'City'
]

target = 'Region'

X = df[features]
y = df[target]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
num_features = ['Sales', 'Delivery_Days']
cat_features = ['Category', 'Sub-Category', 'Segment', 'City']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)


PCA

In [ ]:
pca = PCA(n_components=0.95, random_state=42)

KNN Model Pipeline

In [ ]:
knn_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('pca', pca),
    ('knn', KNeighborsClassifier(
        n_neighbors=7,
        weights='distance',
        metric='euclidean'
    ))
])


Training the Model

In [ ]:
knn_pipeline.fit(X_train, y_train)

Model Evaluation

In [ ]:
y_pred = knn_pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


Confusion matrex

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=knn_pipeline.named_steps['knn'].classes_,
    yticklabels=knn_pipeline.named_steps['knn'].classes_
)
plt.xlabel("Predicted Region")
plt.ylabel("Actual Region")
plt.title("Confusion Matrix ? Region Classification")
plt.savefig("Images/confusion_matrix_region_classification.png", dpi=300)
plt.show()


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

PCA Explained Variance

In [ ]:
pca_model = knn_pipeline.named_steps['pca']
explained_variance = np.cumsum(pca_model.explained_variance_ratio_)

plt.figure()
plt.plot(explained_variance)
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance Curve")
plt.savefig("Images/pca_explained_variance_curve.png", dpi=300)
plt.show()

K Value Tuning

In [ ]:
k_values = range(3, 15)
accuracies = []

for k in k_values:
    model = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('pca', pca),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracies.append(accuracy_score(y_test, preds))


In [ ]:
plt.figure()
plt.plot(k_values, accuracies)
plt.xlabel("K Value")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs K")
plt.savefig("Images/knn_accuracy_vs_k.png", dpi=300)
plt.show()

## City Clustering (Unsupervised Learning)

Objective

Cluster cities based on sales volume, order activity, and delivery efficiency to identify similar market segments for region-aware business and logistics strategies.

### Create City-Level Dataset

In [ ]:
city_df = (
    df.groupby('City')
    .agg(
        Total_Sales=('Sales', 'sum'),
        Avg_Sales=('Sales', 'mean'),
        Total_Orders=('Order ID', 'nunique'),
        Avg_Delivery_Days=('Delivery_Days', 'mean')
    )
    .reset_index()
)

city_df.head()


Select Features for Clustering

In [ ]:
cluster_features = [
    'Total_Sales',
    'Avg_Sales',
    'Total_Orders',
    'Avg_Delivery_Days'
]

X_city = city_df[cluster_features]


Scale the Data

In [ ]:
scaler = StandardScaler()
X_city_scaled = scaler.fit_transform(X_city)

Find Optimal Number of Clusters using Elbow Method

In [ ]:
inertia = []
k_range = range(2, 8)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_city_scaled)
    inertia.append(kmeans.inertia_)


In [ ]:
plt.figure()
plt.plot(k_range, inertia, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.savefig("Images/elbow_method_kmeans.png", dpi=300)
plt.show()

print("Optimal number of clusters appears to be {} based on the elbow method.".format(k_range[np.argmin(np.diff(inertia)) + 1]))

Apply K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
city_df['Cluster'] = kmeans.fit_predict(X_city_scaled)

city_df.head()

Cluster Summary

In [ ]:
cluster_summary = (
    city_df.groupby('Cluster')[cluster_features]
    .mean()
    .round(2)
)

cluster_summary

PCA for 2D Visualization of Clusters

In [ ]:
from sklearn.decomposition import PCA

pca_city = PCA(n_components=2, random_state=42)
X_city_pca = pca_city.fit_transform(X_city_scaled)

city_df['PC1'] = X_city_pca[:, 0]
city_df['PC2'] = X_city_pca[:, 1]

In [ ]:
plt.figure()
sns.scatterplot(
    data=city_df,
    x='PC1',
    y='PC2',
    hue='Cluster',
    palette='Set2'
)
plt.title("City Clusters Based on Sales & Delivery Behavior")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.savefig("Images/city_clusters_pca.png", dpi=300)
plt.show()

#### City clusters align with regional and delivery-time analysis, reinforcing that urban centers exhibit distinct purchasing and logistics behavior.